# Tutorials for Vision Transformer Interpretability

## requirment

In [ ]:
%env CUDA_VISIBLE_DEVICES=0

In [ ]:
from PIL import Image
import torchvision.transforms as transforms
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch
import numpy as np
import cv2
from tqdm import tqdm

from samples.CLS2IDX import CLS2IDX

In [ ]:
normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.ToTensor(),
    normalize,
])

# create heatmap from mask on image
def show_cam_on_image(img, mask):
    heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255
    cam = 0.6 * heatmap + 0.4 * np.float32(img)
    cam = cam / np.max(cam)
    return cam


def print_top_classes(predictions, **kwargs):    
    # Print Top-5 predictions
    prob = torch.softmax(predictions, dim=1)
    class_indices = predictions.data.topk(5, dim=1)[1][0].tolist()
    max_str_len = 0
    class_names = []
    for cls_idx in class_indices:
        class_names.append(CLS2IDX[cls_idx])
        if len(CLS2IDX[cls_idx]) > max_str_len:
            max_str_len = len(CLS2IDX[cls_idx])
    
    print('Top 5 classes:')
    for cls_idx in class_indices:
        output_string = '\t{} : {}'.format(cls_idx, CLS2IDX[cls_idx])
        output_string += ' ' * (max_str_len - len(CLS2IDX[cls_idx])) + '\t\t'
        output_string += 'value = {:.3f}\t prob = {:.1f}%'.format(predictions[0, cls_idx], 100 * prob[0, cls_idx])
        print(output_string)

# heatmap
def heatmap_overlap(img, attribution):
    mask = torch.nn.functional.interpolate(attribution.reshape(1, 1, attribution.shape[0], attribution.shape[1]), scale_factor=16, mode='bilinear')
    mask = mask.squeeze()
    mask = mask.cpu().detach().numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min())
    im = img.permute(1, 2, 0).data.cpu().numpy()
    im = (im - im.min()) / (im.max() - im.min())
    vis = show_cam_on_image(im, mask)
    vis =  np.uint8(255 * vis)
    vis = cv2.cvtColor(np.array(vis), cv2.COLOR_RGB2BGR)
    return vis

## load model

In [ ]:
from baselines.ViT.ViT_new import vit_base_patch16_224

# Model
model = vit_base_patch16_224(pretrained=True).cuda()
model = model.eval()


from baselines.ViT.ViT_LRP import vit_base_patch16_224 as vit_LRP

# Model
model_lrp = vit_LRP(pretrained=True).cuda()
model_lrp = model_lrp.eval()

## Transition Attention Maps

<left>
<img src="./images/pipeline.jpg", width="60%"/>

## load interpretabel methods

In [ ]:
from baselines.ViT.interpret_methods import InterpretTransformer

it = InterpretTransformer(model)

it_lrp = InterpretTransformer(model_lrp)

## methods comparison

### multi-classes

In [ ]:
import glob
import os

n_methods = 4

root_path = os.getcwd()
for img_path in glob.glob(os.path.join(root_path, 'samples', '*.png')):
    img_name = os.path.basename(img_path).replace('.png', '')
    image = Image.open(img_path)
    data = transform(image)
    
    output = model(data.unsqueeze(0).cuda())
    print_top_classes(output)
    
    plt.imshow(image)
    plt.axis('off')
    
    index = np.argmax(output.cpu().data.numpy(), axis=-1)[0]
    class_index = 0
    if index == 243:
        class_index = 282
    elif index == 386 or index == 101:
        class_index = 340
    elif index == 340:
        class_index = 386
    elif index == 282:
        class_index = 243
    elif index == 285:
        class_index = 207
    elif index == 207:
        class_index = 285
    elif index == 161:
        class_index = 87
        
    print("cls1 label: ", index)
    print("cls2 label: ", class_index)
    
    plt.figure(figsize=(n_methods*5, 10))
    
    pred = it.raw_attn(data.unsqueeze(0).cuda())
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach())
    plt.subplot(2, n_methods, 1)
    plt.title('raw_attn')
    plt.imshow(vis)
    plt.axis('off')
    
    target = it.raw_attn(data.unsqueeze(0).cuda(), index=class_index)
    vis = heatmap_overlap(data, target.reshape(14, 14).cpu().detach())
    plt.subplot(2, n_methods, 1 + n_methods)
    plt.title('')
    plt.imshow(vis)
    plt.axis('off')
    
    pred = it.rollout(data.unsqueeze(0).cuda(), start_layer=0)
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach())
    plt.subplot(2, n_methods, 2)
    plt.title('rollout')
    plt.imshow(vis)
    plt.axis('off')
    
    target = it.rollout(data.unsqueeze(0).cuda(), index=class_index, start_layer=0)
    vis = heatmap_overlap(data, target.reshape(14, 14).cpu().detach())
    plt.subplot(2, n_methods, 2 + n_methods)
    plt.title('')
    plt.imshow(vis)
    plt.axis('off')
    
    pred = it_lrp.attribution(data.unsqueeze(0).cuda())
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach()) 
    plt.subplot(2, n_methods, 3)
    plt.title('attribution')
    plt.imshow(vis)
    plt.axis('off')
    
    target = it_lrp.attribution(data.unsqueeze(0).cuda(), index=class_index)
    vis = heatmap_overlap(data, target.reshape(14, 14).cpu().detach()) 
    plt.subplot(2, n_methods, 3 + n_methods)
    plt.title('')
    plt.imshow(vis)
    plt.axis('off')
    
    pred = it.transition_attention_maps(data.unsqueeze(0).cuda(), start_layer=0, steps=20)
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach())
    plt.subplot(2, n_methods, 4)
    plt.title('ours')
    plt.imshow(vis)
    plt.axis('off')
    
    target = it.transition_attention_maps(data.unsqueeze(0).cuda(), index=class_index, start_layer=4, steps=20)
    vis = heatmap_overlap(data, target.reshape(14, 14).cpu().detach()) 
    plt.subplot(2, n_methods, 4 + n_methods)
    plt.title('')
    plt.imshow(vis)
    plt.axis('off')
        
    plt.show()

### single class

In [ ]:
import glob
import os

n_methods = 4

root_path = os.getcwd()
for img_path in glob.glob(os.path.join(root_path, 'samples', '*.jpeg')):
    image = Image.open(img_path)
    data = transform(image)
    
    output = model(data.unsqueeze(0).cuda())
    print_top_classes(output)
    
    plt.imshow(image)
    plt.axis('off')
    
    plt.figure(figsize=(n_methods*5, 10))
    
    pred = it.raw_attn(data.unsqueeze(0).cuda())
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach())
    plt.subplot(1, n_methods, 1)
    plt.title('raw_attn')
    plt.imshow(vis)
    plt.axis('off')
    
    pred = it.rollout(data.unsqueeze(0).cuda(), start_layer=0)
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach())
    plt.subplot(1, n_methods, 2)
    plt.title('rollout')
    plt.imshow(vis)
    plt.axis('off')

    
    pred = it_lrp.attribution(data.unsqueeze(0).cuda())
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach()) 
    plt.subplot(1, n_methods, 3)
    plt.title('attribution')
    plt.imshow(vis)
    plt.axis('off')
    
    pred = it.transition_attention_maps(data.unsqueeze(0).cuda(), start_layer=0, steps=20)
    vis = heatmap_overlap(data, pred.reshape(14, 14).cpu().detach())
    plt.subplot(1, n_methods, 4)
    plt.title('ours')
    plt.imshow(vis)
    plt.axis('off')
    
        
    plt.show()